# Reverse-mode automatic differentiation — Solutions

In [1]:
import numpy as np

## Exercise 1: Verify by hand

In [2]:
a = 4
b = 3

# Forward pass
d = a * (a + b)
print(f"d = {d}")

# Analytical gradients: dd/da = 2a + b, dd/db = a
dd_da = 2 * a + b
dd_db = a
print(f"Analytical: dd/da = {dd_da}, dd/db = {dd_db}")

# Finite difference verification:
# Nudge one variable by eps, recompute d, and divide by eps
eps = 1e-8


def d_func(a, b):
    return a * (a + b)


dd_da_num = (d_func(a + eps, b) - d_func(a, b)) / eps
dd_db_num = (d_func(a, b + eps) - d_func(a, b)) / eps
print(f"Numerical:  dd/da ≈ {dd_da_num:.6f}, dd/db ≈ {dd_db_num:.6f}")

d = 28
Analytical: dd/da = 11, dd/db = 4
Numerical:  dd/da ≈ 11.000000, dd/db ≈ 4.000000


## Exercise 2: Build an autodiff engine

### a) Complete the implementation

In [3]:
from collections import defaultdict


class Variable:
    def __init__(self, value, gradients=None):
        self.value = value
        self._gradients = (
            gradients if gradients is not None else ((self, np.sign(value)),)
        )
        self._stored_gradients = None

    @property
    def gradients(self):
        if self._stored_gradients is None:
            self._stored_gradients = dict(compute_gradients(self))
        return self._stored_gradients


def add(a, b):
    value = a.value + b.value
    gradients = (
        (a, 1),
        (b, 1),
    )
    return Variable(value, gradients)


def mul(a, b):
    value = a.value * b.value
    gradients = (
        (a, b.value),  # d/da (a*b) = b
        (b, a.value),  # d/db (a*b) = a
    )
    return Variable(value, gradients)


def compute_gradients(variable):
    gradients = defaultdict(lambda: 0)

    def _compute_gradients(variable, total_gradient):
        for child_variable, local_gradient in variable._gradients:
            gradient = total_gradient * local_gradient  # chain rule
            gradients[child_variable] += gradient  # sum over paths

            is_leaf = (
                len(child_variable._gradients) == 1
                and child_variable._gradients[0][0] is child_variable
            )
            if not is_leaf:
                _compute_gradients(child_variable, gradient)

    _compute_gradients(variable, total_gradient=1)
    return gradients

### b) Test

In [4]:
a = Variable(4)
b = Variable(3)
c = add(a, b)
d = mul(a, c)

print(f"d = {d.value}")
print(f"dd/da = {d.gradients[a]}")
print(f"dd/db = {d.gradients[b]}")

d = 28
dd/da = 11
dd/db = 4


## Exercise 3: Autodiff for a neuron

In [5]:
from variable import Variable, exp


def loss(y, y_hat):
    return 0.5 * (y - y_hat) ** 2


def sigma(z):
    return 1.0 / (1.0 + exp(-z))


def y_hat(w_0, w_1, x_1):
    return sigma(w_0 + w_1 * x_1)

In [6]:
# --- Verification (just run this cell) ---

# Analytical gradient formulas
def dy_hat_dw_1(w_0, w_1, x_1):
    yh = y_hat(w_0, w_1, x_1)
    return x_1 * yh * (1 - yh)


def dloss_dw_1(y, w_0, w_1, x_1):
    return (y_hat(w_0, w_1, x_1) - y) * dy_hat_dw_1(w_0, w_1, x_1)


# Create variables
x_1 = Variable(0.1, name="x_1")
w_0 = Variable(4, name="w_0")
w_1 = Variable(3, name="w_1")
y = Variable(10, name="y")


def isclose(a, b):
    a = a if not isinstance(a, Variable) else a.value
    b = b if not isinstance(b, Variable) else b.value
    return np.isclose(a, b)


autodiff_grad = loss(y, y_hat(w_0, w_1, x_1)).gradients[w_1]
analytical_grad = dloss_dw_1(y, w_0, w_1, x_1)

assert isclose(autodiff_grad, analytical_grad)
print(f"Autodiff gradient:   {autodiff_grad}")
print(f"Analytical gradient: {analytical_grad}")
print("They match!")

Autodiff gradient:   -0.011904618483381268
Analytical gradient: -0.011904618483381358
They match!
